In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from pynwb import NWBHDF5IO
from IPython.display import display, Markdown

repo = Path(os.environ.get(
    "ANALYSIS_ROOT",
    Path.home() / "Documents" / "Repositories" / "analysis_Belal2026"
))
python_functions = repo / "Python functions"

if str(python_functions) not in sys.path:
    sys.path.insert(0, str(python_functions))

from master_RNAscope import show_block

nwb_root = repo / "NWBdata" / "001832"
nwb_path = nwb_root / "sub-L1-ST8" / "sub-L1-ST8_ses-20240905T115902.nwb"

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

with NWBHDF5IO(str(nwb_path), "r", load_namespaces=True) as io:
    nwbfile = io.read()

    nwb_meta = {
        "session_description": nwbfile.session_description,
        "identifier": nwbfile.identifier,
        "session_start_time": nwbfile.session_start_time,
        "experiment_description": nwbfile.experiment_description,
        "experimenter": list(nwbfile.experimenter) if nwbfile.experimenter is not None else None,
        "lab": nwbfile.lab,
        "institution": nwbfile.institution,
        "protocol": nwbfile.protocol,
        "notes": nwbfile.notes,
        "keywords": list(nwbfile.keywords) if nwbfile.keywords is not None else None,
    }

    subject_meta = {}
    if nwbfile.subject is not None:
        subject_meta = {
            "subject_id": nwbfile.subject.subject_id,
            "description": nwbfile.subject.description,
            "species": nwbfile.subject.species,
            "sex": nwbfile.subject.sex,
            "genotype": nwbfile.subject.genotype,
            "age": nwbfile.subject.age,
            "strain": nwbfile.subject.strain,
        }

    custom_meta = None
    if "session_metadata_custom" in nwbfile.scratch:
        raw_custom = nwbfile.scratch["session_metadata_custom"].data
        if isinstance(raw_custom, bytes):
            raw_custom = raw_custom.decode()
        custom_meta = json.loads(raw_custom)

    counts = (
        nwbfile.processing["rnascope_analysis_metadata"]["experimenter_chrnb2_counts"]
        .to_dataframe()
        .reset_index(drop=True)
    )

display(Markdown(f"# NWB Metadata: `{nwb_path.name}`"))
show_block("NWBFile", nwb_meta)
show_block("Subject", subject_meta)
show_block("Custom", custom_meta)

display(Markdown("## Full Experimenter Counts"))
display(
    counts.sort_values(
        ["condition", "cell_type", "session", "hemisphere", "field_index", "replicate"],
        kind="stable",
    ).reset_index(drop=True)
)

In [ ]:
# to load user stored counts
# the experimeter drew ROIs and counted by eye

from pathlib import Path
import os
import pandas as pd
from pynwb import NWBHDF5IO

repo = Path(os.environ.get(
    "ANALYSIS_ROOT",
    Path.home() / "Documents" / "Repositories" / "analysis_Belal2026"
))
nwb_root = repo / "NWBdata" / "001832"

session_paths = {
    "L1.ST8": nwb_root / "sub-L1-ST8" / "sub-L1-ST8_ses-20240905T115902.nwb",
    "L2.ST8": nwb_root / "sub-L2-ST8" / "sub-L2-ST8_ses-20240909T100245.nwb",
    "L3.ST6": nwb_root / "sub-L3-ST6" / "sub-L3-ST6_ses-20240909T112846.nwb",
    "L4.ST8": nwb_root / "sub-L4-ST8" / "sub-L4-ST8_ses-20240916T101347.nwb",
}

dfs = []

for session, nwb_path in session_paths.items():
    with NWBHDF5IO(str(nwb_path), "r", load_namespaces=True) as io:
        nwbfile = io.read()

        df = (
            nwbfile.processing["rnascope_analysis_metadata"]["experimenter_chrnb2_counts"]
            .to_dataframe()
            .reset_index(drop=True)
        )

    dfs.append(df)

all_user_counts = (
    pd.concat(dfs, ignore_index=True)
    .sort_values(
        ["session", "condition", "cell_type", "hemisphere", "field_index", "replicate"],
        kind="stable",
    )
    .reset_index(drop=True)
)

all_user_counts

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, to_hex

repo = Path(os.environ.get(
    "ANALYSIS_ROOT",
    Path.home() / "Documents" / "Repositories" / "analysis_Belal2026"
))
sys.path.insert(0, str(repo / "Python functions"))

from master_functions import boxplot_rtype_plotly

ndnf = all_user_counts[all_user_counts["cell_type"].eq("NDNF+")].copy()

intact = ndnf[ndnf["condition"].eq("Intact")].copy()
lesioned = ndnf[ndnf["condition"].eq("Lesioned")].copy()

r_viridis_stops = ["#440154", "#3B528B", "#21918C", "#5DC963", "#FDE725"]
cmap = LinearSegmentedColormap.from_list("r_boxplot_viridis", r_viridis_stops)

fields = list(ndnf["field"].drop_duplicates())
field_colors = {
    field: to_hex(cmap(i / max(len(fields) - 1, 1)))
    for i, field in enumerate(fields)
}

fig = boxplot_rtype_plotly(
    [intact["count"].values, lesioned["count"].values],
    positions=[1, 2],
    xrange=[0.5, 2.5],
    yrange=[0, 60],
    widths=0.25,
    tick_labels=["Intact", "Lesioned"],
    showpoints=True,
    jitter_frac=0.6,
    point_diameter=7,
    point_alpha=0.6,
    point_color=[
        intact["field"].map(field_colors).tolist(),
        lesioned["field"].map(field_colors).tolist(),
    ],
    point_edgecolor="gray",
    title="NDNF+ RNAscope user counts",
    yaxis_title="CHRNB2 puncta count",
    width=450,
    height=500,
)

SAVE = False
SVG_DIR = repo / "Paper analysis" / "Figure 11" / "svg"

if SAVE:
    SVG_DIR.mkdir(parents=True, exist_ok=True)
    svg_path = SVG_DIR / "NDNF_user_counts_boxplot.svg"
    fig.write_image(str(svg_path), format="svg")
    print(f"Exported: {svg_path}")

fig.show()


In [ ]:
th = all_user_counts[all_user_counts["cell_type"].eq("TH+")].copy()

intact = th[th["condition"].eq("Intact")].copy()
lesioned = th[th["condition"].eq("Lesioned")].copy()

fields = list(th["field"].drop_duplicates())
field_colors = {
    field: to_hex(cmap(i / max(len(fields) - 1, 1)))
    for i, field in enumerate(fields)
}

fig = boxplot_rtype_plotly(
    [intact["count"].values, lesioned["count"].values],
    positions=[1, 2],
    xrange=[0.5, 2.5],
    yrange=[0, 20],
    widths=0.25,
    tick_labels=["Intact", "Lesioned"],
    showpoints=True,
    jitter_frac=0.6,
    point_diameter=7,
    point_alpha=0.6,
    point_color=[
        intact["field"].map(field_colors).tolist(),
        lesioned["field"].map(field_colors).tolist(),
    ],
    point_edgecolor="gray",
    title="TH+ RNAscope user counts",
    yaxis_title="CHRNB2 puncta count",
    width=450,
    height=500,
)

fig.show()

if SAVE:
    svg_path = SVG_DIR / "TH_user_counts_boxplot.svg"
    fig.write_image(str(svg_path), format="svg")
    print(f"Exported: {svg_path}")